NLP Preprocessing & Sentiment Labelling
### HPDP Project 2 — Real-Time Sentiment Analysis

**Input:** `youtube_comments_raw.csv` (from Notebook 1)

**Output:** `youtube_comments_cleaned.csv` — fully preprocessed + labelled, ready for model training.

---
### Full NLP Pipeline
```
Raw Text
   ↓  1. Basic Cleaning     (remove URLs, emails, numbers, special chars)
   ↓  2. Lowercasing        (normalize case)
   ↓  3. Tokenization       (split into words)
   ↓  4. Stop Word Removal  (remove 'the', 'is', 'a' ...)
   ↓  5. Stemming           (running → run, better → better)
   ↓  6. Lemmatization      (runs → run, better → good)
   ↓  7. Sentiment Label    (TextBlob → positive / neutral / negative)
Cleaned Text + Label
```

## Step 1 — Install & Import Libraries

In [ ]:
!pip install textblob nltk --quiet
print('✅ Packages installed!')

✅ Packages installed!


In [ ]:
import nltk

# Download all required NLTK resources
nltk.download('punkt',            quiet=True)
nltk.download('punkt_tab',        quiet=True)
nltk.download('stopwords',        quiet=True)
nltk.download('wordnet',          quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
print('✅ NLTK resources downloaded!')

✅ NLTK resources downloaded!


In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet

# Initialize NLP tools
tokenizer  = RegexpTokenizer(r'\b\w{3,}\b')   # Words with 3+ letters only
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Output settings
OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = f'{OUTPUT_DIR}/youtube_comments_cleaned.csv'

print('✅ All imports successful!')
print(f'   Stop words loaded  : {len(stop_words)} words')

✅ All imports successful!
   Stop words loaded  : 198 words


## Step 2 — Upload Raw CSV
Upload the `youtube_comments_raw.csv` file downloaded from Notebook 1.

In [ ]:
from google.colab import files

print('📂 Please upload your youtube_comments_raw.csv...')
uploaded = files.upload()

uploaded_filename = list(uploaded.keys())[0]
df = pd.read_csv(uploaded_filename)

print(f'\n✅ Loaded: {uploaded_filename}')
print(f'   Rows    : {len(df):,}')
print(f'   Columns : {list(df.columns)}')
df.head()

📂 Please upload your youtube_comments_raw.csv...


Saving youtube_comments_raw.csv to youtube_comments_raw.csv

✅ Loaded: youtube_comments_raw.csv
   Rows    : 18,805
   Columns : ['comment_id', 'parent_id', 'comment_type', 'author', 'text', 'like_count', 'reply_count', 'published_at', 'video_id']


,comment_id,parent_id,comment_type,author,text,like_count,reply_count,published_at,video_id
0,UgylxmmbT-YO8_OFn6l4AaABAg,NaN,top_level,@GeographyNow,What's that? Your country only has ONE King?.....,6100,310,2018-06-07T00:00:00+00:00,dV-H1EKmCxA
1,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D8h97X3H60Qa,UgylxmmbT-YO8_OFn6l4AaABAg,reply,@Bluebeanzz,Technically is 8 sultan and 1 raja (Perlis)…,53,2,2018-06-07T00:00:00+00:00,dV-H1EKmCxA
2,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D97vuIuj3y_1,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D8h97X3H60Qa,reply,@Darklord-ms1if,@Bluebeanzz well that's my village,0,0,2020-06-07T00:00:00+00:00,dV-H1EKmCxA
3,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D9D88cnHcori,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D8h97X3H60Qa,reply,@hideriplays2626,"@Bluebeanzz 7 sultans, 1 raja (Perlis), and 1...",0,0,2021-06-07T00:00:00+00:00,dV-H1EKmCxA
4,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D8h94bhH_dvC,UgylxmmbT-YO8_OFn6l4AaABAg,reply,@allysuke,Why didn’t u talk about the F1 circuit,144,2,2018-06-07T00:00:00+00:00,dV-H1EKmCxA


## Step 3 — Basic Cleaning
Remove noise: URLs, emails, numbers, emojis, special characters.

In [ ]:
def basic_clean(text):
    """Remove URLs, emails, numbers, emojis, special characters."""
    if not isinstance(text, str) or text.strip() == '':
        return ''
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove emojis and special characters (keep only letters, spaces, #, @)
    text = re.sub(r'[^\w\s#@_/-]', '', text)
    # Normalize whitespace and lowercase
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text


print('🧹 Step 3: Basic cleaning...')
df['step1_cleaned'] = df['text'].apply(basic_clean)

# Preview
print('\nBefore vs After cleaning (sample):')
sample = df[df['step1_cleaned'].str.len() > 5][['text', 'step1_cleaned']].head(3)
for _, row in sample.iterrows():
    print(f'  RAW     : {str(row["text"])[:80]}')
    print(f'  CLEANED : {str(row["step1_cleaned"])[:80]}')
    print()

🧹 Step 3: Basic cleaning...

Before vs After cleaning (sample):
  RAW     : What's that? Your country only has ONE King?... Pssshhh Okay that's cute. We hav
  CLEANED : whats that your country only has one king pssshhh okay thats cute we have nine t

  RAW     : Technically is 8 sultan and 1 raja (Perlis)…
  CLEANED : technically is sultan and raja perlis

  RAW     : @Bluebeanzz  well that's my village
  CLEANED : @bluebeanzz well thats my village



## Step 4 — Tokenization
Split text into individual words (tokens). Only keeps words with 3+ letters.

In [ ]:
def tokenize(text):
    """Split text into list of word tokens (3+ letters only)."""
    if not text:
        return []
    return tokenizer.tokenize(text)


print('✂️  Step 4: Tokenizing...')
df['step2_tokens'] = df['step1_cleaned'].apply(tokenize)

print('\nTokenization sample:')
for _, row in df[df['step2_tokens'].str.len() > 3].head(3).iterrows():
    print(f'  TEXT   : {row["step1_cleaned"][:60]}')
    print(f'  TOKENS : {row["step2_tokens"][:8]}')
    print()

✂️  Step 4: Tokenizing...

Tokenization sample:
  TEXT   : whats that your country only has one king pssshhh okay thats
  TOKENS : ['whats', 'that', 'your', 'country', 'only', 'has', 'one', 'king']

  TEXT   : technically is sultan and raja perlis
  TOKENS : ['technically', 'sultan', 'and', 'raja', 'perlis']

  TEXT   : @bluebeanzz well thats my village
  TOKENS : ['bluebeanzz', 'well', 'thats', 'village']



## Step 5 — Stop Word Removal
Remove common words that carry no sentiment meaning: `the`, `is`, `a`, `and`, etc.

In [ ]:
def remove_stopwords(tokens):
    """Remove English stop words from token list."""
    return [word for word in tokens if word not in stop_words]


print('🛑 Step 5: Removing stop words...')
print(f'   Stop words in use: {len(stop_words)} (e.g. {list(stop_words)[:8]})')

df['step3_no_stopwords'] = df['step2_tokens'].apply(remove_stopwords)

# Show how many words were removed
total_before = df['step2_tokens'].str.len().sum()
total_after  = df['step3_no_stopwords'].str.len().sum()
print(f'\n   Words before : {total_before:,}')
print(f'   Words after  : {total_after:,}')
print(f'   Removed      : {total_before - total_after:,} stop words ({(total_before-total_after)/total_before:.1%})')

print('\nStop word removal sample:')
for _, row in df[df['step3_no_stopwords'].str.len() > 3].head(3).iterrows():
    print(f'  BEFORE : {row["step2_tokens"][:8]}')
    print(f'  AFTER  : {row["step3_no_stopwords"][:8]}')
    print()

🛑 Step 5: Removing stop words...
   Stop words in use: 198 (e.g. ["wasn't", 'been', 'when', "needn't", 't', 'while', 'but', "they've"])

   Words before : 203,887
   Words after  : 152,350
   Removed      : 51,537 stop words (25.3%)

Stop word removal sample:
  BEFORE : ['whats', 'that', 'your', 'country', 'only', 'has', 'one', 'king']
  AFTER  : ['whats', 'country', 'one', 'king', 'pssshhh', 'okay', 'thats', 'cute']

  BEFORE : ['technically', 'sultan', 'and', 'raja', 'perlis']
  AFTER  : ['technically', 'sultan', 'raja', 'perlis']

  BEFORE : ['bluebeanzz', 'well', 'thats', 'village']
  AFTER  : ['bluebeanzz', 'well', 'thats', 'village']



## Step 6 — Stemming
Reduce words to their root form using Porter Stemmer.

`running → run` | `movies → movi` | `amazing → amaz`

> Note: Stemming is aggressive and may produce non-real words. That is normal and expected.

In [ ]:
def stem_tokens(tokens):
    """Apply Porter Stemming to each token."""
    return [stemmer.stem(word) for word in tokens]


print('🌱 Step 6: Stemming...')
df['step4_stemmed'] = df['step3_no_stopwords'].apply(stem_tokens)

print('\nStemming sample (before → after):')
for _, row in df[df['step4_stemmed'].str.len() > 3].head(3).iterrows():
    pairs = list(zip(row['step3_no_stopwords'][:6], row['step4_stemmed'][:6]))
    print(f'  {" | ".join([f"{a} → {b}" for a,b in pairs])}')

🌱 Step 6: Stemming...

Stemming sample (before → after):
  whats → what | country → countri | one → one | king → king | pssshhh → pssshhh | okay → okay
  technically → technic | sultan → sultan | raja → raja | perlis → perli
  bluebeanzz → bluebeanzz | well → well | thats → that | village → villag


## Step 7 — Lemmatization
Reduce words to their dictionary base form using WordNet Lemmatizer.

`running → run` | `better → good` | `movies → movie`

> Lemmatization is smarter than stemming — it produces real words. We use POS tagging to improve accuracy.

In [ ]:
def get_wordnet_pos(word):
    """Map POS tag to WordNet POS for better lemmatization."""
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {
        'J': wordnet.ADJ,
        'V': wordnet.VERB,
        'N': wordnet.NOUN,
        'R': wordnet.ADV
    }
    return tag_map.get(tag, wordnet.NOUN)


def lemmatize_tokens(tokens):
    """Apply lemmatization with POS tagging to each token."""
    return [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in tokens]


print('📚 Step 7: Lemmatizing... (this may take a moment)')
df['step5_lemmatized'] = df['step3_no_stopwords'].apply(lemmatize_tokens)

print('\nLemmatization sample (before → after):')
for _, row in df[df['step5_lemmatized'].str.len() > 3].head(3).iterrows():
    pairs = list(zip(row['step3_no_stopwords'][:6], row['step5_lemmatized'][:6]))
    print(f'  {" | ".join([f"{a} → {b}" for a,b in pairs])}')

📚 Step 7: Lemmatizing... (this may take a moment)

Lemmatization sample (before → after):
  whats → whats | country → country | one → one | king → king | pssshhh → pssshhh | okay → okay
  technically → technically | sultan → sultan | raja → raja | perlis → perlis
  bluebeanzz → bluebeanzz | well → well | thats → thats | village → village


## Step 8 — Join Tokens Back to String
Convert the final token list back into a single string for model input.

In [ ]:
print('🔗 Step 8: Joining tokens to final cleaned text...')

# Final cleaned text uses lemmatized tokens
df['cleaned_text'] = df['step5_lemmatized'].apply(lambda tokens: ' '.join(tokens))

# Remove rows where cleaned text is too short
before = len(df)
df = df[df['cleaned_text'].str.len() > 5].reset_index(drop=True)
print(f'   Removed {before - len(df):,} rows (too short after cleaning)')
print(f'   Remaining: {len(df):,} comments')

print('\nFull pipeline preview (raw → final):')
for _, row in df.head(3).iterrows():
    print(f'  RAW   : {str(row["text"])[:80]}')
    print(f'  FINAL : {row["cleaned_text"][:80]}')
    print()

🔗 Step 8: Joining tokens to final cleaned text...
   Removed 1,332 rows (too short after cleaning)
   Remaining: 17,473 comments

Full pipeline preview (raw → final):
  RAW   : What's that? Your country only has ONE King?... Pssshhh Okay that's cute. We hav
  FINAL : whats country one king pssshhh okay thats cute nine thanks many malay geograpeep

  RAW   : Technically is 8 sultan and 1 raja (Perlis)…
  FINAL : technically sultan raja perlis

  RAW   : @Bluebeanzz  well that's my village
  FINAL : bluebeanzz well thats village



## Step 9 — Sentiment Labelling with TextBlob

TextBlob scores polarity from **-1.0** (very negative) to **+1.0** (very positive):
- `polarity > 0.1` → **positive**
- `polarity < -0.1` → **negative**  
- otherwise → **neutral**

> TextBlob labels are used as **training labels** for your Naive Bayes and LSTM models in Notebook 3.

In [ ]:
def get_polarity(text):
    if not text:
        return 0.0
    return round(TextBlob(text).sentiment.polarity, 4)


def get_sentiment(text):
    if not text:
        return 'neutral'
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0.1:
        return 'positive'
    elif polarity < -0.1:
        return 'negative'
    else:
        return 'neutral'


print('🏷️  Step 9: Labelling sentiments...')
df['polarity']  = df['cleaned_text'].apply(get_polarity)
df['sentiment'] = df['cleaned_text'].apply(get_sentiment)

print('\n✅ Sentiment labelling done!')
print('\nDistribution:')
print(df['sentiment'].value_counts())
print('\nAs percentages:')
print(df['sentiment'].value_counts(normalize=True).map('{:.1%}'.format))

🏷️  Step 9: Labelling sentiments...

✅ Sentiment labelling done!

Distribution:
sentiment
neutral     11091
positive     4886
negative     1496
Name: count, dtype: int64

As percentages:
sentiment
neutral     63.5%
positive    28.0%
negative     8.6%
Name: proportion, dtype: object


In [ ]:
# Sanity check — sample per label
print('🟢 POSITIVE EXAMPLES:')
print(df[df['sentiment'] == 'positive']['cleaned_text'].head(3).to_string())

print('\n🔴 NEGATIVE EXAMPLES:')
print(df[df['sentiment'] == 'negative']['cleaned_text'].head(3).to_string())

print('\n⚪ NEUTRAL EXAMPLES:')
print(df[df['sentiment'] == 'neutral']['cleaned_text'].head(3).to_string())

🟢 POSITIVE EXAMPLES:
0     whats country one king pssshhh okay thats cute...
8     geography rakyat hidup full blood malaysian pr...
17                    joelvoerman maybe cause indo live

🔴 NEGATIVE EXAMPLES:
12    brendanbomantheswagnificent noits chinese mean...
13      brendanbomantheswagnificent hate malaysia right
77    nine ring give rule middle earth south east as...

⚪ NEUTRAL EXAMPLES:
1                       technically sultan raja perlis
2                        bluebeanzz well thats village
3    bluebeanzz sultan raja perlis yang dipertuan b...


## Step 10 — Save Final CSV & Download
Only keep relevant columns for the output file.

In [ ]:
final_columns = [
    'comment_id',
    'comment_type',
    'author',
    'cleaned_text',   # Final preprocessed text (model input)
    'polarity',       # TextBlob polarity score
    'sentiment',      # Label: positive / neutral / negative
    'like_count',
    'reply_count',
    'published_at',
]

final_columns = [c for c in final_columns if c in df.columns]
df_final = df[final_columns].copy()

df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'✅ Cleaned CSV saved → {OUTPUT_CSV}')
print(f'   Rows    : {len(df_final):,}')
print(f'   Columns : {list(df_final.columns)}')
print(f'   Size    : {os.path.getsize(OUTPUT_CSV)/1024:.1f} KB')

df_final.head()

✅ Cleaned CSV saved → /content/output/youtube_comments_cleaned.csv
   Rows    : 17,473
   Columns : ['comment_id', 'comment_type', 'author', 'cleaned_text', 'polarity', 'sentiment', 'like_count', 'reply_count', 'published_at']
   Size    : 2814.8 KB


,comment_id,comment_type,author,cleaned_text,polarity,sentiment,like_count,reply_count,published_at
0,UgylxmmbT-YO8_OFn6l4AaABAg,top_level,@GeographyNow,whats country one king pssshhh okay thats cute...,0.3833,positive,6100,310,2018-06-07T00:00:00+00:00
1,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D8h97X3H60Qa,reply,@Bluebeanzz,technically sultan raja perlis,0.0000,neutral,53,2,2018-06-07T00:00:00+00:00
2,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D97vuIuj3y_1,reply,@Darklord-ms1if,bluebeanzz well thats village,0.0000,neutral,0,0,2020-06-07T00:00:00+00:00
3,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D9D88cnHcori,reply,@hideriplays2626,bluebeanzz sultan raja perlis yang dipertuan b...,0.0000,neutral,0,0,2021-06-07T00:00:00+00:00
4,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D8h94bhH_dvC,reply,@allysuke,didnt talk circuit,0.0000,neutral,144,2,2018-06-07T00:00:00+00:00


In [ ]:
print('⬇️  Downloading cleaned CSV...')
files.download(OUTPUT_CSV)
print('✅ Done! Check your Downloads folder.')
print('\n➡️  Next: Use youtube_comments_cleaned.csv in for model training.')

⬇️  Downloading cleaned CSV...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done! Check your Downloads folder.

➡️  Next: Use youtube_comments_cleaned.csv in for model training.


In [ ]:
print('=' * 55)
print('  NLP PREPROCESSING SUMMARY')
print('=' * 55)
print(f"  Step 1 - Basic Cleaning   : ✅ URLs, emails, numbers removed")
print(f"  Step 2 - Lowercasing      : ✅ All text normalized to lowercase")
print(f"  Step 3 - Tokenization     : ✅ Words with 3+ letters kept")
print(f"  Step 4 - Stop Word Removal: ✅ {len(stop_words)} stop words removed")
print(f"  Step 5 - Stemming         : ✅ Porter Stemmer applied")
print(f"  Step 6 - Lemmatization    : ✅ WordNet Lemmatizer + POS tagging")
print(f"  Step 7 - Sentiment Labels : ✅ TextBlob polarity")
print('─' * 55)
print(f"  Total comments            : {len(df_final):,}")
print(f"  Positive                  : {(df_final['sentiment']=='positive').sum():,} ({(df_final['sentiment']=='positive').mean():.1%})")
print(f"  Neutral                   : {(df_final['sentiment']=='neutral').sum():,} ({(df_final['sentiment']=='neutral').mean():.1%})")
print(f"  Negative                  : {(df_final['sentiment']=='negative').sum():,} ({(df_final['sentiment']=='negative').mean():.1%})")
print(f"  Avg polarity score        : {df_final['polarity'].mean():.4f}")
print('=' * 55)
print('\n✅ Dataset is ready for Model Training!')

  NLP PREPROCESSING SUMMARY
  Step 1 - Basic Cleaning   : ✅ URLs, emails, numbers removed
  Step 2 - Lowercasing      : ✅ All text normalized to lowercase
  Step 3 - Tokenization     : ✅ Words with 3+ letters kept
  Step 4 - Stop Word Removal: ✅ 198 stop words removed
  Step 5 - Stemming         : ✅ Porter Stemmer applied
  Step 6 - Lemmatization    : ✅ WordNet Lemmatizer + POS tagging
  Step 7 - Sentiment Labels : ✅ TextBlob polarity
───────────────────────────────────────────────────────
  Total comments            : 17,473
  Positive                  : 4,886 (28.0%)
  Neutral                   : 11,091 (63.5%)
  Negative                  : 1,496 (8.6%)
  Avg polarity score        : 0.0841

✅ Dataset is ready for Model Training!
